In [1]:
%cd ../
from uparxive.xml_to_json.xml_to_dense_text import *
import xml.etree.ElementTree as ET
output_xml_path='debug/test.xml'

/nas/zhangtianning.di/projects/unique_data_build


In [2]:
from tqdm.auto import tqdm

In [3]:
#with open(output_xml_path, 'wb') as f:tree.write(f, pretty_print=True, xml_declaration=True, encoding='UTF-8')

In [4]:
import os
verbose = False
filte_out_note = True
reterive_result_mode = False
use_count_type_ref= False

In [23]:
tmp_xml_path = '/nvme/zhangtianning/datasets/whole_arxiv_data/whole_arxiv_all_files/unprocessed_xml/1907.11741/Main.clean.xml'
reterive_result_mode = bool(reterive_result_mode)
_paper_id = os.path.basename(os.path.dirname(tmp_xml_path))
paper_id = f"ArXiv.{_paper_id}"
#paper_id = identify_string_type(_paper_id.replace('_',"/"))
parser = etree.XMLParser(remove_comments=True)
with open(tmp_xml_path) as f:
    tree = etree.parse(f, parser)  # get tree of XML hierarchy

discard_note(tree)
ref_count = retreive_all_cite(tree)
new_tree = deepcopy(tree)
new_tree,reference_labels,bibitem_ref_metadata,note_ref_labels, note_ref_metadata= remove_entire_bibliography_and_build_labels(new_tree,verbose=verbose, filte_out_note=filte_out_note)
if len(note_ref_metadata)>5:
    for key,val in note_ref_metadata.items():
        logging.info(f"{key} ==> [ {better_latex_sentense_string(' '.join(val[1].itertext()))} ]")
    checkTooManyNote(f'the note_ref_metadata num={len(note_ref_metadata) } is too much , please check the file {tmp_xml_path}')
    logging.warning('WARNING:Too Many note, we roll back to no note mode')
    tree,reference_labels,bibitem_ref_metadata,note_ref_labels, note_ref_metadata= remove_entire_bibliography_and_build_labels(tree,verbose=verbose, filte_out_note=False)
else:
    tree = new_tree
# print(print_namespace_tree(reference_labels))
# print("============================================")
# print(print_namespace_tree(note_ref_labels))
# raise
#ref_count = retreive_all_cite(tree)
### now, we need divide the ref dict into two part, 1. The ref used in the tex 2. The ref note used in the tex
reference_labels, reference_labels_not_in_context = divide_the_dict_into_two_part_by_keys(reference_labels, ref_count)
note_ref_labels, note_ref_labels_not_in_context = divide_the_dict_into_two_part_by_keys(note_ref_labels,ref_count)
bibitem_ref_metadata, bibitem_ref_metadata_not_in_context = divide_the_dict_into_two_part_by_keys(bibitem_ref_metadata,ref_count)
note_ref_metadata, note_ref_metadata_not_in_context = divide_the_dict_into_two_part_by_keys(note_ref_metadata,ref_count)

#if len(bibitem_ref_metadata)>0, f"Error: this file [{tmp_xml_path}] donts have bib???"

tree,put_back_keys = put_note_string_back_into_each_sentence(tree,ref_count,note_ref_metadata)
# notice, after this line, the key in note_ref_metadata and note_ref_labels is different

tree,figures_labels, figures_metadata = remove_figures_record_the_labels(tree)
tree,tables_labels, tables_metadata   = remove_tables_record_the_labels(tree)
tree,equation_labels                  = remove_tags_and_record_the_labels(tree,'equation')

tree,equation_group_labels            = remove_tags_and_record_the_labels(tree,'equationgroup')
tree, labels = remove_tags_and_record_alllabels(tree, exclude_keys=['figures','figure','tables','table','bibref','equation','equationgroup'])

all_citation_keys = set(ref_count.keys())
all_reference_keys= (set(reference_labels.keys())|
                     set(note_ref_labels.keys())|
                     set(figures_labels.keys())|
                     set(tables_labels.keys())|
                     set(equation_labels.keys())|
                     set(equation_group_labels.keys()))

for val_pool in labels.values():
    all_reference_keys = all_reference_keys | set(val_pool)
missing_citation = all_citation_keys - all_reference_keys
missing_citation_labels = {missing_citation_label:f'MissingCite_{i}' for i,missing_citation_label in enumerate(missing_citation)}


#assert len(bibitem_ref_metadata)>0, f"Error: this file [{tmp_xml_path}] donts have bib???"

if reterive_result_mode:
    ReferenceDir= os.path.join(output_dir, "Reference")
    assert os.path.exists(os.path.join(ReferenceDir,'reference.keys.done'))
    assert os.path.getsize(os.path.join(ReferenceDir,'reference.txt')) == 0, "if you want to inject the reterive result, please make sure all the element is reterived"
    with open(os.path.join(ReferenceDir,'reference.keys.done'),'r') as f:
        reference_keys = [t.strip() for t in f]

    with open(os.path.join(ReferenceDir,'reference.es_retrived_citation.json.done'),'r') as f:
        reference_reterives = json.load(f)
    assert len(reference_keys) == len(reference_reterives), "the reterive result should have the same length as the keys"
    new_label_mapping = {}
    for key, reterive_result in zip(reference_keys,reference_reterives):
        if key not in new_label_mapping:new_label_mapping[key] = []
        new_label_mapping[key].append(get_unique_id_from_reterive_result(reterive_result))
    for key in new_label_mapping.keys():
        new_label_mapping[key] = "<"+ ",".join(new_label_mapping[key]) + ">"
    reference_labels = new_label_mapping


whole_ref_to_labels = collect_whole_reference({
    'Figure':figures_labels,
    'Table':tables_labels,
    'Reference':reference_labels,
    'Equation':equation_labels,
    'Equationgroup':equation_group_labels,
    'Missing':missing_citation_labels
    }|labels, use_count_type_ref=use_count_type_ref)

lack_ref = list(set(ref_count) - (set(all_reference_keys)|set(whole_ref_to_labels)))
#print(set(ref_count))
#print(set(all_reference_keys)|set(whole_ref_to_labels))
if len(lack_ref)>0:
    logging.info(f'you have {len(lack_ref)} ref lacks, such as {lack_ref[:4]}, please check the file {tmp_xml_path}')
    raise MisMatchRefError

## now, the left note metadata is those string looks like a citation, and we will put them back into the bibitem information
for remain_key, remain_val in note_ref_metadata.items():
    reference_labels[remain_key]=note_ref_labels[remain_key]
    string = cleanup_reference_string(remain_val[1], whole_ref_to_labels,paper_id, refs_that_wont_recovery=put_back_keys)
    bibitem_ref_metadata[remain_key]=better_latex_sentense_string(string)



whole_ref_to_labels = collect_whole_reference({

        'Figure':figures_labels,
        'Table':tables_labels,
        'Reference':reference_labels,
        'Equation':equation_labels,
        'Equationgroup':equation_group_labels,
        'Missing':missing_citation_labels
         }|labels, use_count_type_ref=use_count_type_ref)
tree = cleanup_xml(tree, whole_ref_to_labels,paper_id, put_back_keys)
tree = replace_item_block_with_markdown_format(tree)
tree, abstract     = remove_entire_partition_and_collect(tree,'abstract')
tree, acknowledge  = remove_entire_partition_and_collect(tree,'acknowledgements')

In [24]:
with open(output_xml_path, 'wb') as f:tree.write(f, pretty_print=True, xml_declaration=True, encoding='UTF-8')

In [18]:
def replace_item_block_with_markdown_format(item):
    """
        <itemize xml:id="S5.I1">
            <item xml:id="S5.I1.i1">
              <tags>
                <tag>•</tag>
                <tag role="autoref">item </tag>
                <tag role="typerefnum">1st item</tag>
              </tags>
              <para xml:id="S5.I1.i1.p1">
                <p>Eighty-two percent of users were aged 18 to 34 years.</p>
              </para>
            </item>
            <item xml:id="S5.I1.i2">
              <tags>
                <tag>•</tag>
                <tag role="autoref">item </tag>
                <tag role="typerefnum">2nd item</tag>
              </tags>
              <para xml:id="S5.I1.i2.p1">
                <p>Seventy-five percent of participants reported that they used Twitter at least once a day, whereas 62% of all participants stated that they used it several times a day.</p>
              </para>
            </item>
            <item xml:id="S5.I1.i3">
              <tags>
                <tag>•</tag>
                <tag role="autoref">item </tag>
                <tag role="typerefnum">3rd item</tag>
              </tags>
              <para xml:id="S5.I1.i3.p1">
                <p>What option best describes what you use Twitter for?: The most popular answer (with a frequency of 51%) among 10 options was that participants used Twitter to keep up with or share the news in general.</p>

              </para>
            </item>
          </itemize>

        to
        <itemize xml:id="S5.I1">
            <p>
            - Eighty-two percent of users were aged 18 to 34 years.
            - Seventy-five percent of participants reported that they used Twitter at least once a day, whereas 62% of all participants stated that they used it several times a day.
            - What option best describes what you use Twitter for?: The most popular answer (with a frequency of 51%) among 10 options was that participants used Twitter to keep up with or share the news in general.
            </p>
        </itemize>
        
    """
    ns = {'default': 'http://dlmf.nist.gov/LaTeXML'}
    elements = item.findall('.//default:item', ns)
    for element in elements:
        tag = element.find('.//default:tags', ns)
        prefix="- "
        if tag is not None:
            prefix = tag.find('.//default:tag', ns)
            if prefix is not None:
                prefix = f"{prefix.text} "
            tag.getparent().remove(tag)
        ### if tag is a number then add this number at begining
        
        #all_p = element.findall('.//default:p', ns)
        ### merge all p content into one
        new_p = etree.Element('{%s}p' % ns['default'])
        new_p.text = prefix + better_latex_sentense_string(" ".join(element.itertext()))
        element.getparent().replace(element, new_p)
    return item

In [11]:
collect_whole_section_into_one_paper(tree)

[{'tag': 'I',
  'section_title': 'Introduction',
  'section_content': [['Social media users are constantly creating content that connects them to others, but are users aware of the emotional influence that social media has on their moods or lives? While consuming and sharing information online is advantageous to connecting people, it may pose a risk to ignore the emotions that we (consciously or unconsciously) spread out to hundreds or thousands of people with a single post.'],
   ['Over the past few years, emotional contagion through social media has been of great interest. Kramer et al. [Ref.[MissingCite_0] of ArXiv.1907.11741] argue, “Emotional states can be transferred to others via emotional contagion, leading people to experience the same emotions without their awareness . Emotional contagion is well established in laboratory experiments, with people transferring positive and negative emotions to others.” In addition, several studies, e.g., (See [Ref.[MissingCite_0,MissingCite_13

In [140]:
is_start_of_string = not left.strip() or left.strip()[-1] in {'.', '!', '?', ';'}
print(is_start_of_string)

True


In [141]:
left="This is record to."
position = re.match(r'\b({})\b'.format('|'.join(prepositions)), left.strip().split()[-1])

In [15]:
note_ref_metadata

{}